# Hướng dẫn Tinh chỉnh (Fine-tuning) PaddleOCR Recognition cho Mã Container trên Google Colab

Notebook này hỗ trợ bạn kết nối Google Drive, giải nén và tự động cắt (crop) dữ liệu từ tập ảnh thô dựa trên nhãn tọa độ, tự động chia bộ dữ liệu Train/Val theo tỉ lệ 90/10, cấu hình và huấn luyện tinh chỉnh mô hình nhận dạng ký tự **PP-OCRv3** của PaddleOCR chuyên biệt cho phông chữ Container, sau đó xuất ra mô hình suy luận tĩnh (`inference model`) lưu về Google Drive của bạn.

## Bước 1: Kết nối Google Drive và GPU

In [ ]:
# 1. Kết nối tới Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

# 2. Kiểm tra thông tin GPU
!nvidia-smi

## Bước 2: Cài đặt thư viện PaddlePaddle GPU và PaddleOCR

In [ ]:
# 1. Cài đặt thư viện PaddlePaddle hỗ trợ GPU (phù hợp với CUDA trên Colab)
!pip install paddlepaddle-gpu

# 2. Clone mã nguồn PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 3. Cài đặt các thư viện phụ thuộc
!pip install -r requirements.txt

## Bước 3: Tự động Giải nén và Xử lý cắt (Crop) dữ liệu ảnh container

Đoạn mã dưới đây sẽ tự động thực hiện:
1. Giải nén tệp `ContainerNum_dataset.zip` từ Drive của bạn.
2. Giải nén các tệp zip lồng nhau (`train_images.zip`, `train_images_label.zip`).
3. Đọc nhãn tọa độ (`x1, y1, x2, y2, nhãn_chữ`) từ các tệp `.txt` gán nhãn.
4. Tự động cắt (crop) ảnh chứa dòng mã container từ ảnh thô lớn.
5. Tự động chia tập dữ liệu thành 90% Train và 10% Validation.
6. Tạo các tệp nhãn `rec_train_label.txt` và `rec_val_label.txt` chuẩn định dạng của PaddleOCR.

In [ ]:
import os
import zipfile
import io
import cv2
import re
import random

# 1. Đường dẫn file ZIP trên Google Drive
outer_zip_path = "/content/drive/MyDrive/ContainerNum_dataset.zip"

print("Đang giải nén file ZIP chính...")
with zipfile.ZipFile(outer_zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_dataset")

# 2. Đường dẫn các zip con vừa được giải nén ra
train_images_zip = "/content/temp_dataset/ContainerNum_dataset/train_images.zip"
train_labels_zip = "/content/temp_dataset/ContainerNum_dataset/train_images_label.zip"

print("Đang giải nén ảnh thô và nhãn tọa độ...")
with zipfile.ZipFile(train_images_zip, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_train_images")

with zipfile.ZipFile(train_labels_zip, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_train_labels")

# 3. Tạo cấu trúc thư mục huấn luyện
os.makedirs("/content/PaddleOCR/train_data/rec/train", exist_ok=True)
os.makedirs("/content/PaddleOCR/train_data/rec/val", exist_ok=True)

# Quét tất cả file nhãn dạng text
label_dir = "/content/temp_train_labels/images_label"
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]

# Chia tập Train (90%) và Val (10%)
random.seed(42)
random.shuffle(label_files)
split_idx = int(len(label_files) * 0.9)
train_files = label_files[:split_idx]
val_files = label_files[split_idx:]

def process_and_crop(files, subset):
    label_entries = []
    count = 0
    for lf in files:
        base_name = os.path.splitext(lf)[0]
        img_name = base_name + ".jpg"
        
        img_path = os.path.join("/content/temp_train_images/images", img_name)
        label_path = os.path.join(label_dir, lf)
        
        if not os.path.exists(img_path):
            continue
            
        # Đọc ảnh
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        # Đọc nhãn tọa độ dạng x1,y1,x2,y2,label
        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        for idx, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 5:
                try:
                    # Ép kiểu tọa độ
                    x1 = int(float(parts[0]))
                    y1 = int(float(parts[1]))
                    x2 = int(float(parts[2]))
                    y2 = int(float(parts[3]))
                    label = ",".join(parts[4:]).upper()
                    
                    # Làm sạch nhãn chữ (chỉ giữ ký tự và số)
                    label = re.sub(r'[^A-Z0-9]', '', label)
                    if not label:
                        continue
                        
                    h_orig, w_orig, _ = img.shape
                    x1_c, y1_c = max(0, x1), max(0, y1)
                    x2_c, y2_c = min(w_orig, x2), min(h_orig, y2)
                    
                    if x2_c <= x1_c or y2_c <= y1_c:
                        continue
                        
                    # Cắt vùng chữ
                    crop = img[y1_c:y2_c, x1_c:x2_c]
                    
                    # Lưu ảnh crop
                    crop_name = f"{base_name}_{idx}.jpg"
                    save_path = os.path.join(f"/content/PaddleOCR/train_data/rec/{subset}", crop_name)
                    cv2.imwrite(save_path, crop)
                    
                    # Tạo chuỗi nhãn chuẩn
                    relative_path = f"train_data/rec/{subset}/{crop_name}"
                    label_entries.append(f"{relative_path}\t{label}\n")
                    count += 1
                except Exception as e:
                    pass
                    
    # Ghi file danh sách nhãn
    label_txt_path = f"/content/PaddleOCR/train_data/rec_{subset}_label.txt"
    with open(label_txt_path, 'w', encoding='utf-8') as f:
        f.writelines(label_entries)
    print(f"-> Phân tập '{subset}': Đã cắt và lưu thành công {count} ảnh mã container.")

process_and_crop(train_files, "train")
process_and_crop(val_files, "val")

# Dọn dẹp bộ nhớ tạm giải phóng không gian ổ đĩa Colab
!rm -rf /content/temp_dataset /content/temp_train_images /content/temp_train_labels
print("Đã dọn dẹp các thư mục trung gian thành công!")

## Bước 4: Tải Trọng số Pre-trained Model PP-OCRv3 English

In [ ]:
# Tải mô hình pre-trained PP-OCRv3 tiếng Anh
!mkdir -p pretrain_models
!wget -P pretrain_models/ https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar

# Giải nén gói trọng số
%cd pretrain_models
!tar -xf en_PP-OCRv3_rec_train.tar
%cd ..

## Bước 5: Cấu hình File YAML để bắt đầu Training

Đoạn mã Python dưới đây tự động cập nhật các trường đường dẫn dữ liệu huấn luyện, đường dẫn trọng số pre-trained, số epoch chạy và cấu hình từ điển vào tệp config `en_PP-OCRv3_rec.yml` của PaddleOCR.

In [ ]:
import yaml

config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 1. Ghi đè các thông số cấu hình cốt lõi dùng đường dẫn tuyệt đối
config['Global']['pretrained_model'] = '/content/PaddleOCR/pretrain_models/en_PP-OCRv3_rec_train/best_accuracy'
config['Global']['save_model_dir'] = '/content/PaddleOCR/output/v3_rec_container/'
config['Global']['epoch_num'] = 150              # Huấn luyện 150 Epochs
config['Global']['print_batch_step'] = 10
config['Global']['use_gpu'] = True

# 2. Cấu hình đường dẫn dữ liệu Training
config['Train']['dataset']['data_dir'] = '/content/PaddleOCR/train_data/rec/train/'
config['Train']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_train_label.txt']

# 3. Cấu hình đường dẫn dữ liệu Evaluation
config['Eval']['dataset']['data_dir'] = '/content/PaddleOCR/train_data/rec/val/'
config['Eval']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_val_label.txt']

# Lưu lại cấu hình mới vào đè lên file
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, default_flow_style=False)

print("Đã ghi đè cấu hình file YAML huấn luyện thành công!")

## Bước 6: Khởi chạy Huấn luyện Tinh chỉnh (Fine-tuning)

In [ ]:
# Chạy lệnh python huấn luyện dùng đường dẫn tuyệt đối
!python /content/PaddleOCR/tools/train.py -c /content/PaddleOCR/configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml

## Bước 7: Xuất Mô hình Suy luận tĩnh (Inference Model) về Google Drive

Khi quá trình training kết thúc, tệp trọng số tốt nhất được lưu tại `/content/PaddleOCR/output/v3_rec_container/best_accuracy`. Ta sẽ xuất nó sang định dạng suy luận tĩnh rồi lưu trực tiếp vào Google Drive để bạn dễ dàng tải về tích hợp vào code chạy ứng dụng Web UI offline!

In [ ]:
# 1. Định nghĩa thư mục lưu trữ trên Google Drive của bạn
drive_output_dir = '/content/drive/MyDrive/paddle_rec_inference/'

# 2. Chạy script export model của PaddleOCR dùng đường dẫn tuyệt đối
!python /content/PaddleOCR/tools/export_model.py \
  -c /content/PaddleOCR/configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml \
  -o Global.pretrained_model=/content/PaddleOCR/output/v3_rec_container/best_accuracy \
  Global.save_inference_dir={drive_output_dir}

print(f"Xuất mô hình thành công! Bạn có thể lấy tệp tại thư mục: {drive_output_dir} trên Google Drive.")